# Imports

In [ ]:
import numpy as np 
import pandas as pd
from mggp import MGGP
from utils.utils import *

# Execute

In [2]:
save_model = "mggp_models/dissertacao_best_model_FR1.pkl"
OUTPUT_FEATURES = 3

# Main Train

In [ ]:
u_data = []
y_data = []

# for tire_position in ["FR", "FL", "RR", "RL"]:
for tire_position in ["FR"]:
    number = '35_25'
    folder = 'Job1_2023_07_29_12_15_29_Mix'
    path = f"iTire road test data/{folder}/Tire_Features_extraidas_{number}.csv"
    
    u_train, y_train, _, _ = load_data(path, OUTPUT_FEATURES, folder, tire=tire_position)
    u_train, y_train = media_movel_pandas(u_train), media_movel_pandas(y_train)

    folder = 'Job1_2023_07_29_11_36_05_Mix'
    path = f"iTire road test data/{folder}/Tire_Features_extraidas_{number}.csv"
    u_train1, y_train1, _, _ = load_data(path, OUTPUT_FEATURES, folder, tire=tire_position)
    u_train1, y_train1 = u_train1[1250:2000, :], y_train1[1250:2000, :] # Slalom in 45kph, 2 times
    u_train1, y_train1 = media_movel_pandas(u_train1), media_movel_pandas(y_train1)

    u_train = np.concatenate([u_train, u_train1], axis=0)
    y_train = np.concatenate([y_train, y_train1], axis=0)

    u_data.append(u_train)
    y_data.append(y_train)

u_combined = np.concatenate(u_data, axis=0)
y_combined = np.concatenate(y_data, axis=0)

mggp = MGGP(inputs=u_combined,
            outputs=y_combined,
            validation=(u_combined, y_combined),
            nDelays=1,
            generations=100,
            populationSize=250,
            evaluationMode="MSE",
            k=100,
            evaluationType='MShooting',
            evaluationTypeTest="FreeRun",
            nTerms=20,
            maxHeight=2,
            mutationRate=0.2,
            crossoverRate=0.9,
            elitePercentage=10,
            filename = "mggp_models/mggp_test.pkl",
            mode="NARX"
            )

mggp.run()